# AI Engineering Interview Prep
## Section 21 — Scenario-Based GenAI System Design (Top 100 Questions)

**Mangesh Jha | Senior Software Developer → AI Engineer / Architect**

---
> 💡 **How to use:** Use `Ctrl+F` to search any scenario. 🏷️ markers indicate Mangesh's real project experience (Nyaya-Pro, JobPilot AI, ExamGenie AI, and Yotta Infrastructures operational excellence) used as grounding in candidate responses.
> 
> **Principal AI Engineer/Interviewer Perspective:** Every question is answered in the voice of a candidate interviewing at OpenAI, Google DeepMind, Anthropic, Microsoft, Amazon, Meta, or Nvidia. Core deep dives include the comprehensive 21-part scenario-based system design format with ASCII architecture diagrams.


In [ ]:
print("📚 Section 21: Top 100 Scenario-Based GenAI System Design Interview Questions (2026)")
print("✅ Grounded in Nyaya-Pro, JobPilot AI, ExamGenie AI, and Yotta scale operations!")


---
## 🗂️ Section 1: RAG System Design (Questions 1–20)


### Q1. Design a RAG chatbot for 10 million enterprise documents.
**A:**
Here is the production-grade end-to-end design for this system, leveraging my experience at Yotta Infrastructures for large-scale operations and Nyaya-Pro's retrieval optimization.

#### 1. Clarifying Questions
- What is the average document size? (Assume 10 pages / ~4000 words per document).
- What are the latency requirements? (Under 1.5 seconds Time-to-First-Token).
- What is the update frequency? (Real-time updates, ~10,000 document additions/updates per day).
- Do we need to handle access control? (Yes, role-based access control (RBAC) at the document level).

#### 2. Requirements Gathering
- 10 million documents * 4,000 words ≈ 40 billion words.
- 40 billion words ≈ 53 billion tokens.
- 53 billion tokens chunked into 512-token segments (with 10% overlap) ≈ 115 million chunks.

#### 3. Functional Requirements
- Semantic query search and conversational answering over the entire corpus.
- Citation support (attributing output to document IDs and page numbers).
- Role-based access control (RBAC) enforcement.

#### 4. Non-Functional Requirements
- P95 Latency: < 2.0s total response generation.
- Uptime: 99.99% high availability.
- Security: Zero data leakage between tenants/roles.

#### 5. Capacity Estimation
- 115 million vectors. If using a 1536-dimensional model (like OpenAI `text-embedding-3-small`) in float32:
  - Size per vector = 1536 * 4 bytes = 6 KB.
  - Raw vector memory = 115M * 6 KB ≈ 690 GB.
  - HNSW Index overhead (approx 20%) ≈ 138 GB.
  - Total RAM required for vector indexing in memory ≈ 828 GB.
- Storage for text chunks and metadata ≈ 1.2 TB (SSD).

#### 6. High-Level Architecture
```
+-----------------------------------------------------------------------+
|                          Ingestion Pipeline                           |
|  Docs -> Kafka -> Spark/Ray Workers -> Embeddings -> Pinecone Vector  |
|                                                   DB (with RBAC tags) |
+-----------------------------------------------------------------------+
                                    ^
                                    |
+-----------------------------------------------------------------------+
|                           Online Pipeline                             |
|  User Query -> FastAPI -> Keycloak JWT -> Hybrid Search (dense/BM25)  |
|     -> Reranker (Cross-Encoder) -> LLM (Gemini Flash) -> User Stream  |
+-----------------------------------------------------------------------+
```

#### 7. Component Design
- **Ingestion Engine:** Async Spark pipeline reads from enterprise stores, extracts text using layout-aware parsers, and splits into semantic chunks.
- **Embedding Generator:** Triton Inference Server hosting locally deployed `bge-large-en-v1.5` running on NVIDIA L4 GPUs.
- **Vector Database:** Managed Pinecone or sharded Milvus with HNSW indexing.
- **Orchestration Layer:** FastAPI backend running in Kubernetes.

#### 8. Data Flow
1. User query sent to FastAPI containing Keycloak authorization token.
2. FastAPI extracts user roles and generates query vector.
3. FastAPI queries Vector Database using a metadata filter: `tenant_id == user.tenant_id AND role IN user.roles`.
4. Top 50 candidates are retrieved.
5. Candidates are sent to a local cross-encoder reranking service (`ms-marco-MiniLM-L-6-v2`), returning top 5.
6. Top 5 chunks are injected into the context block of LLM.
7. LLM streams the citation-anchored response via Server-Sent Events (SSE).

#### 9. Database Design
- PostgreSQL metadata DB containing tables: `documents` (id, title, tenant_id, required_roles, version), `users` (id, tenant_id, roles).

#### 10. Vector Database Design
- Document vectors indexed with metadata tags: `{"tenant_id": "T101", "allowed_roles": ["admin", "finance"], "doc_id": "D99"}`.

#### 11. Agent Design (Not applicable here - direct pipeline).

#### 12. RAG Design
- Parent-Document retriever pattern: Chunks are small (512 tokens), but when a match is found, the parent context (1024 tokens) is sent to the LLM for generation.

#### 13. Prompt Engineering Strategy
- System Prompt containing strict grounding: "You are an enterprise assistant. Answer the user's question using ONLY the retrieved documents. Cite your sources by using document title and page number. If the answer is not in the documents, state that you cannot answer."

#### 14. Evaluation Framework
- RAGAS metrics tracked daily using a synthetic evaluation set: Faithfulness, Answer Relevance, Context Precision, and Context Recall.

#### 15. Security & Guardrails
- Input/Output guardrails via Llama Guard to screen prompts for leakage attempts and verify output against PII leakage patterns.

#### 16. Scalability Design
- Horizontal Pod Autoscaler (HPA) for FastAPI pods based on CPU/Request count. Vector database split into read replicas across multiple regions.

#### 17. Cost Optimization
- Local Triton embedding serving to avoid API cost on ingestion. Gemini 2.5 Flash as LLM for extremely cheap context processing.

#### 18. Monitoring & Observability
- Prometheus for latency metrics. OpenTelemetry + Arize Phoenix for RAG tracing (tracking retrieval vs generation times).

#### 19. Tradeoffs
- **Vector DB (Pinecone vs Milvus):** Chose Pinecone for low operational overhead, but Milvus self-hosted on Yotta's GPU bare-metal would save cost if server scale allows.
- **Embedding Model (Local vs API):** Local L4 GPUs have higher upfront setup but save $10K+/month in API tokens for 10M documents.

#### 20. Interviewer Follow-up Questions
- How do you handle cold starts on read replicas when new documents are added?
- What if a document classification changes dynamically?

#### 21. Final 2-Minute Interview Answer
"To build a RAG chatbot for 10 million enterprise documents, I propose a production-grade decoupled ingestion-serving architecture. We process documents into 115 million vectors. To handle this scale securely and within a 2-second latency budget, we deploy a local Triton server on L4 GPUs for embeddings, store vectors in an RBAC-filtered vector index (like Pinecone/Milvus), and use a local cross-encoder reranker. The generation uses Gemini 2.5 Flash for high context windows and low cost. Having built Nyaya-Pro with FAISS and citation support, and optimized real-time dashboards at Yotta using Keycloak/NgRx, I know how to scale this secure enterprise retrieval flow from day one."

### Q2. User asks a question but retrieval returns irrelevant chunks. How would you debug?
**A:**
🏷️ *This is the exact debugging loop I developed for Nyaya-Pro when it struggled with legal terminology context shifts.*

#### 1. Clarifying Questions
- Is the query using domain-specific jargon?
- Are we using dense vector search only, or hybrid search?
- What embedding model is being used?

#### 2. Debugging Steps Flowchart
```
+--------------------------------------------------------------+
| Identify Failed Query -> Run Query Isolation                 |
+--------------------------------------------------------------+
                               |
                               v
+--------------------------------------------------------------+
| Compare Vector Cosine Similarity vs. Keyword Match (BM25)    |
+--------------------------------------------------------------+
       /                                                \
      v                                                  v
Similarity low?                                    Jargon issue?
  -> Inadequate chunk size or overlap.               -> Fine-tune Embedding / add BM25.
```

#### 3. Root Cause Investigation
- **Embedding Model Drift/Mismatch:** Verify if the query embedding model is identical to the document embedding model.
- **Lack of Keyword Matching:** Pure vector search fails at exact-match terms (like section numbers, part IDs, e.g., 'Section 302' or code names). Fix: Integrate hybrid search (BM25 + Dense vector cosine similarity).
- **Sub-optimal Chunking:** The target text is buried in a huge paragraph (diluted vector) or split across two chunks (broken sentence). Fix: Use recursive/semantic chunking with parent-child relationships.
- **Query Transformation Failure:** The user query is too conversational. Fix: Run Query Reformulation / Query Expansion (HyDE) before embedding.

#### 4. Post-Mortem Actions
- Add the failed query to the Golden Test Set.
- Evaluate retriever precision using MRR (Mean Reciprocal Rank) and NDCG metrics.
- Re-tune the reranker weights (e.g. adjust hybrid search alpha coefficient: dense_score * alpha + sparse_score * (1-alpha)).

### Q3. Design a legal document assistant with citation support.
**A:**
🏷️ *This design represents Nyaya-Pro, my agentic legal RAG assistant developed to navigate the transition between the Indian Penal Code (IPC) and the new Bharatiya Nyaya Sanhita (BNS).*

#### 1. Clarifying Questions
- Do we need to support comparative analysis between old and new laws? (Yes, e.g., mapping IPC 302 to BNS 101).
- What format are the legal documents in? (OCR PDFs of court cases, Gazette notifications, structured act PDFs).

#### 2. Functional Requirements
- Answer legal queries citing specific sections, clauses, and page numbers.
- Comparative legal lookup (IPC vs BNS map).
- Source attribution validation.

#### 3. Non-Functional Requirements
- Faithfulness score (RAGAS): > 93%.
- Output structure validity: 100% JSON compliance (crucial for UI rendering).

#### 4. High-Level Architecture
```
+------------+      +---------------------+      +---------------------+
| User Query | ---> | Agentic Query Router| ---> | Criminal Code Index |
+------------+      +---------------------+      +---------------------+
                               |                             |
                               v                             v
                    +---------------------+      +---------------------+
                    | comparative mapping |      | FAISS Vector DB     |
                    +---------------------+      +---------------------+
                               |                             |
                               +-------------->--------------+
                                             |
                                             v
                               +-----------------------------+
                               | Cross-Encoder Reranker      |
                               +-----------------------------+
                                             |
                                             v
                               +-----------------------------+
                               | Structured Gemini 2.5 Flash |
                               +-----------------------------+
```

#### 5. Vector Database Design (FAISS)
- Index partitioned by Act (Constitution, BNS, IPC). Every vector contains metadata:
  ```json
  { "act": "BNS", "section": "101", "subsection": "2", "page_no": 45, "raw_text": "..." }
  ```

#### 6. RAG Design & Citation Verification
- **Query Routing:** LLM routing layer identifies if the query refers to a criminal trial, corporate law, or comparative maps. Filters search scope.
- **Citation Answering Loop:** The prompt instructs the model to return a structured JSON response:
  ```json
  { "answer": "Murder is penalized under BNS Section 101...", 
    "citations": [ { "act": "BNS", "section": "101", "exact_quote": "Whoever commits murder..." } ] }
  ```
- **Verification Engine:** A Python post-processor extracts the citation and does a substring lookup of `exact_quote` within the matching segment of the document vector metadata. If not found, the citation is flagged and omitted.

#### 7. Prompt Engineering
- The prompt contains the exact schemas and strict instruction: "Write down the citation BEFORE formulating the final answer to ground your reasoning."

#### 8. Tradeoffs
- **Reranker Latency vs Citation Accuracy:** Reranking adds 200ms of latency, which I solved by rendering a skeleton screen showing the 'Retrieving and verifying laws...' phase in the Angular frontend. This trade-off is critical because a hallucinated law can result in severe legal liabilities.

#### 9. Final 2-Minute Interview Answer
"In Nyaya-Pro, I designed a citation-grounded RAG system for legal documents. The key is strict validation. We ingestion-index Gazette PDFs, chunking them with overlapping parent-child relationships and metadata tagging (Act, Section, Page). In the online phase, an agentic query router directs the query to target indexes. We use a hybrid search and cross-encoder reranking approach to isolate the top 3 chunks. The LLM is prompted to extract citation quotes before answering. A post-processing step validates that the cited quotes exist verbatim in our source metadata. This design achieved a 94% faithfulness score on RAGAS, ensuring production-grade legal reliability."

### Q4. How would you reduce hallucinations in a financial RAG system?
**A:**
- **Clarification & Requirements:** Needs absolute mathematical and factual correctness over balance sheets, earnings calls, and financial filings.
- **Design & Mitigation Strategy:**
  1. **Strict Context Gating:** Prompt system instructions enforce: "If the exact number is not in the context, do not calculate or output it. Respond with: 'Information not found in target filings'."
  2. **Numerical Extraction Parser:** Parse tabular PDF data using specialized table extractors (e.g., pdfplumber) into Markdown tables. Do not convert tables to flat text.
  3. **Self-Correction Guardrails:** Run an output validator checking if numerical figures in the LLM's generated response match the source documents using simple Python lookup dictionaries.
- **Tradeoffs:** Lower recall and lower creativity for 100% numerical accuracy. Better to refuse an answer than to hallucinate a financial figure.

### Q5. Design a customer support AI using company knowledge base.
**A:**
- **Clarification & Requirements:** Automate support queries, integrate with ticketing APIs, escalate to human agents when resolution fails.
- **Architecture & Data Flow:**
  - Input message → Router classifies intent (billing, troubleshooting, FAQ).
  - FAQ queries hit semantic cache (Redis) or Pinecone KB.
  - Response generated with Gemini Flash, scanned by Llama Guard.
  - Escalation mechanism triggers if user sentiment turns negative or if retrieval confidence is low.
- **Grounding:** Similar to the OneYotta portal support flow, utilizing NgRx for UI state transitions during escalations and Keycloak JWT for auth mapping.

### Q6. Chunk size is causing poor answers. How would you optimize it?
**A:**
- **Clarification & Requirements:** Optimize context retrieval precision when large chunks introduce noise and small chunks lose vital surrounding context.
- **Optimization Strategy:**
  1. **Semantic Chunking:** Instead of fixed-token boundaries, split text based on semantic transitions (using cosine distance of sentence embeddings).
  2. **Parent-Child Chunking:** Index small vectors (128-token child chunks) for high retrieval precision, but return the broader parent context (512-1024 tokens) to the LLM during generation.
  3. **Overlap Optimization:** Tune chunk overlap to 15-20% to prevent context loss on boundaries.

### Q7. Design a multilingual RAG system.
**A:**
- **Clarification & Requirements:** Support queries in English, Hindi, Spanish, and French over documents written in any language.
- **Architecture & Ingestion:**
  - Use a native multilingual embedding model (e.g., `Cohere-embed-multilingual-v3.0` or `multilingual-e5-large`) mapping all languages to a shared vector space.
  - Store documents in their original language. Query in any language directly retrieves matching semantic nodes.
  - Translate LLM outputs locally or instruct the generative LLM (Gemini) to output in the user's source query language.

### Q8. How would you implement Hybrid Search (BM25 + Vector Search)?
**A:**
- **Clarification & Requirements:** Combine exact keyword matches (e.g., product IDs, legal section numbers) with semantic matching.
- **Implementation Details:**
  - Query is sent to BM25 Engine (Elasticsearch/Qdrant) and Vector DB in parallel.
  - Normalise scores from both search engines using Reciprocal Rank Fusion (RRF):
    `Score(d) = 1 / (60 + Rank_dense(d)) + 1 / (60 + Rank_sparse(d))`
  - Select top-k documents based on RRF scores, then pass to the cross-encoder reranker.
- **Grounding:** Used in Nyaya-Pro to ensure that queries referencing legal sections (e.g., '101') returned exact matches alongside semantic equivalents.

### Q9. Design a GraphRAG architecture for connected knowledge.
**A:**
Here is the design for GraphRAG, mapping structured relationships across documents.

#### 1. Ingestion Pipeline
```
Docs -> Entity Extraction (LLM) -> Node/Relation Triplet (Entity1 -> Relation -> Entity2)
                                -> Graph Database (Neo4j) + Vector Index
```

#### 2. High-Level Architecture
```
User Query -> Entity Extractor -> (1) Search Neo4j Graph for local subgraphs
                               -> (2) Dense Search Vector DB
                               -> Combined context (subgraph triplets + text chunks)
                               -> LLM Answering
```

#### 3. Database & Graph Design
- Graph DB: Neo4j containing entities (e.g. `Person`, `Law`, `Company`) and relationships (e.g. `MUTATES_TO`, `OWNED_BY`).
- Vector DB: FAISS/Pinecone indexing raw entities for entity linking.

#### 4. Tradeoffs
- GraphRAG handles complex multi-hop questions (e.g. 'Show all laws updated by amendment X affecting tenant Y') that normal RAG fails at, but at the cost of 5× higher ingestion complexity and cost (requires heavy LLM calls for extraction).

### Q10. How would you handle document updates in real-time?
**A:**
- **Design & Ingestion Flow:**
  - Event-driven ingestion using Kafka. Document edits trigger a message.
  - Consuming worker retrieves the document, deletes existing vectors associated with `document_id` metadata tag, re-chunks, re-embeds, and upserts new vectors.
- **Caching Strategy:** Invalidate semantic cache entries containing references to the mutated document.

### Q11. Design a RAG system for healthcare records.
**A:**
- **Critical Requirements:** HIPAA compliance, PII de-identification, and medical taxonomy embedding.
- **Design Strategy:**
  1. **PII Scrubbing:** Local gateway running `Presidio` or custom NER de-identifies patients (replacing names/SSNs with unique keys) before sending queries to cloud LLMs.
  2. **Medical Indexing:** Embed documents using clinical-specific embedding models (e.g., `PubMedBERT` or clinical-T5) mapping to medical taxonomies (UMLS, SNOMED).
  3. **Audit Logging:** Logs tracking every document access, matching patient authorization tokens.

### Q12. User asks questions across multiple PDFs. How would you preserve context?
**A:**
- **Design Strategy:**
  1. **Session State Memory:** Store historical conversation summaries in a Redis cache associated with the session.
  2. **Hierarchical Indexing:** Extract PDF-level metadata and construct a dynamic table-of-contents guide to route query attention across PDF scopes.
  3. **Context Window Expansion:** Load the context blocks of active PDFs into Gemini 2.5 Flash's 1M-token context window with prefix prompt caching to minimize cost.
- **Grounding:** Grounded in ExamGenie AI's multi-page PDF processing stack.

### Q13. Design a RAG pipeline for scanned PDFs and images.
**A:**
- **Architecture:**
  - Document → Layout-aware parser (e.g., `Unstructured` or `Azure Document Intelligence`).
  - Run OCR (Tesseract / EasyOCR) on images to extract textual tables.
  - Convert images/figures to text summaries using a Vision-Language Model (VLM like Gemini 2.5 Flash).
  - Index OCR text + VLM summaries into the Vector DB alongside original image chunks.

### Q14. Retrieval latency exceeds 2 seconds. How would you optimize?
**A:**
- **Optimization Strategy:**
  1. **Index Quantization:** Apply Scalar Quantization (SQ8) or Product Quantization (PQ) to vectors in Pinecone/FAISS to reduce memory size and speed up ANN calculation.
  2. **Prefiltering:** Ensure metadata filters (e.g. `tenant_id`) are indexed to shrink search space before vector math.
  3. **Asynchronous Reranking:** Run reranker on top-10 instead of top-50, and use vLLM backend on L4 GPUs for low latency.
- **Grounding:** Optimized Nyaya-Pro latency from 1.5s to 600ms by reducing reranking candidate pool size and leveraging FAISS index optimizations.

### Q15. Design a RAG system with source attribution.
**A:**
- **Design Strategy:**
  - Every chunk in vector DB contains fields: `{"doc_id": "X", "source_url": "...", "page": 12}`.
  - Prompt specifies structured output: `{ "answer": "...", "sources": ["doc_id"] }`.
  - UI extracts `sources` and displays interactive citation cards (like Nyaya-Pro's citation component, built in Angular 18).

### Q16. How would you evaluate retrieval quality?
**A:**
- **Evaluation Framework:**
  - Retrieve top-k chunks on a Golden Dataset of queries.
  - Measure: **Context Recall** (are all ground-truth facts retrieved?), **Context Precision** (are the most relevant chunks ranked at the top?), and **MRR** (Mean Reciprocal Rank).
  - Run these automated evals in CI/CD using RAGAS/LangSmith before deploying retriever prompt or model updates.

### Q17. Design a RAG system supporting 1M daily users.
**A:**
- **Scale Architecture:**
  - Read-heavy architecture: Deploy multiple read replicas of Vector DB across geographical regions.
  - Route requests using Cloudflare Load Balancing.
  - Implement semantic caching using Redis to serve identical queries instantly, bypassing LLM/Vector DB.
  - Rate-limit using sliding window algorithm in FastAPI.
- **Grounding:** Grounded in Yotta's API orchestration and scaling best practices.

### Q18. How would you prevent confidential data leakage?
**A:**
- **Mitigation Strategy:**
  1. **Strict Metadata Isolation:** Pre-filter every search query at database level (never retrieve then filter in memory).
  2. **PII Masking Layer:** Run an inline tokenizer scrubbing SSNs, emails, and credentials before saving vectors or calling LLM APIs.
  3. **Adversarial Red-Teaming:** Inject jailbreak prompts in tests to verify that the system refuses to output other users' metadata.

### Q19. Design semantic caching for a RAG application.
**A:**
- **Architecture:**
  - Query → Embed → Vector search on Redis semantic cache (similarity threshold e.g., > 0.96).
  - If match found → return cached response instantly (latency < 50ms).
  - If mismatch → run standard RAG pipeline, cache the query + response with a 24h TTL.
- **Grounding:** Implemented in Nyaya-Pro, reducing API costs by 35% on repetitive legal questions.

### Q20. Long-context model vs RAG: which would you choose and why?
**A:**
- **Decision Framework:**
  - **Choose RAG when:** Document set is vast (>10M tokens), real-time updates are needed, cost per query must be kept low, or role-based security is required.
  - **Choose Long-Context (Gemini 1.5 Pro) when:** The task requires reasoning over a single, dense context (e.g., a codebase, a video, a single thick book), or multi-hop retrieval across all pages is complex.
- **Tradeoffs:** RAG is cheap and fast but can miss details. Long-context is highly accurate but expensive and slow.

---
## 🤖 Section 2: AI Agent System Design (Questions 21–40)


### Q21. Design an autonomous travel planning agent.
**A:**
Here is the design for an autonomous agent executing flight searches, hotel bookings, and itinerary generation.

#### 1. Clarifying Questions
- Does the agent make real financial transactions? (Yes, requires OAuth confirmation / human-in-the-loop validation).
- What API backends do we connect to? (Amadeus API for flights, Expedia for hotels, Google Maps for locations).

#### 2. Functional Requirements
- Search and compare flights and lodging options based on user constraints.
- Compile a day-by-day travel itinerary.
- Execute bookings with confirmation alerts.

#### 3. Non-Functional Requirements
- Budget constraint compliance: 100% strict limit enforcement.
- Security: Secure vault storage for user booking credentials.

#### 4. High-Level Architecture
```
User Input -> ReAct Orchestrator (LangGraph state machine)
                     |
        +------------+------------+
        |                         |
        v                         v
 Flight Search Tool         Hotel Booking Tool
 (Amadeus API)              (Expedia API)
        |                         |
        +------------+------------+
                     |
                     v
        Human-in-the-loop Gate (Angular Approval Dialog)
                     |
                     v
        Execution Engine -> Email/SMS Notification
```

#### 5. State Management & Agent Memory
- LangGraph state schema tracking: `{"user_id": str, "budget": float, "current_plan": dict, "flight_status": str, "hotel_status": str, "approved_by_user": bool}`.
- Session checkpointing stored in PostgreSQL for 'time-travel' rollback state.

#### 6. Tool Design
- Define tools using Pydantic models with clear descriptions:
  ```python
  class FlightSearchInput(BaseModel):
      origin: str = Field(description="3-letter airport code (e.g. BOM)")
      destination: str = Field(description="3-letter destination airport (e.g. DXB)")
      date: str = Field(description="Date in YYYY-MM-DD format")
  ```

#### 7. Prompt Engineering
- System Prompt enforces ReAct loop control: "You have access to flight and hotel search tools. Always check flights first, calculate budget headroom, search hotels, construct the itinerary, and request final human approval before booking."

#### 8. Security & Guardrails
- Strictly prevent the agent from executing a booking without checking the `approved_by_user` flag. If the model attempts to call `confirm_booking` without previous user approval, throw a tool validation error.

#### 9. Tradeoffs
- **Autonomous vs. Structured Flow:** Structured workflows are much more reliable than open-ended ReAct loops. I chose a structured state graph (LangGraph) over a free CrewAI agent to guarantee the system doesn't make random API queries.

#### 10. Final 2-Minute Interview Answer
"To design an autonomous travel agent, I propose a LangGraph state machine structure for predictable execution. The agent is defined by nodes (Planner, Flight Search, Hotel Booking, and Reviewer) with explicit transitions. To handle payment security, I implement a Human-in-the-loop node that halts the graph execution, stores state, and triggers an Angular UI approval dialog. My experience with multi-agent CrewAI pipelines in JobPilot AI and token streaming allows me to orchestrate this stateful flow with high reliability and cost efficiency."

### Q29. Design a multi-step reasoning agent for tax advisory.
**A:**
🏷️ *This design mirrors my agentic legal work in Nyaya-Pro, adapting legal RAG pipelines to mathematical and statutory reasoning.*

#### 1. Clarifying Questions
- Does the advisor calculate tax liability or just provide citations? (Both: needs to compute liabilities and cite tax codes).
- How do we handle changes in tax laws? (Integrate RAG over the IRS / Income Tax Department database).

#### 2. High-Level Architecture
```
User Query -> Planner Agent -> Retain Tax Code Docs (RAG)
                           -> Math Executor (Python Sandbox) -> Fact Checker
                           -> Structured Report -> User
```

#### 3. Execution & Safety Design
- **Step 1 (Plan):** Break query into rules (e.g. capital gains vs income).
- **Step 2 (Retrieve):** Semantic + exact search for relevant tax codes (BM25 + Dense vector match over FAISS index).
- **Step 3 (Execute):** LLMs are bad at math. We instruct the model to write python calculation scripts, run them in a sandboxed Docker container, and return execution outputs.
- **Step 4 (Verify):** Verify mathematical calculations using a separate validation agent comparing outputs against raw tax code rules.

#### 4. Prompt Engineering
- Strict guidelines instructing the model to list equations and legal citations before printing values.

#### 5. Tradeoffs
- Running code sandboxes adds latency (1.5s startup cost), but is mandatory because letting an LLM do raw multi-step calculations is a recipe for hallucinations and IRS non-compliance.

### Q30. Design an AI project manager agent.
**A:**
🏷️ *This utilizes the multi-agent orchestration principles I developed for JobPilot AI to parse inputs, suggest optimizations, and schedule actions.*

#### 1. Clarifying Questions
- What ticketing systems do we write to? (Jira, GitHub Issues, Trello).
- How does the agent interact? (Slack app, Discord, or direct web UI).

#### 2. Architecture & Data Flow
```
Slack Event -> FastAPI webhook -> LLM Classifier -> (1) Update Ticket Status
                                                -> (2) Draft Sprint Summary
                                                -> (3) Alert blockers to dev
```
- **Task Routing:** Parse conversations using a supervisor agent. When 'X blocker' is detected, route to assignee's channel.
- **Integration:** CRUD operations executed on Jira API using Pydantic schema validation.
- **Loop Prevention:** Strictly enforce a stateless processing cycle on event triggers to avoid cyclical updates (e.g., agent updating Jira triggering a webhook which triggers the agent again).

#### 3. Tradeoffs
- Chose direct event-driven FastAPI endpoints over polling. Saves server cost and keeps execution real-time, but requires robust queue handling (Celery/RabbitMQ) to scale when multiple events trigger simultaneously.

### Q22. Design an AI recruiter agent that screens candidates.
**A:**
- **Clarification & Requirements:** Automatically evaluate candidate resumes against job descriptions (JobPilot AI core workflow) and score fit.
- **Design Strategy:**
  1. **Resume Processing:** Angular UI uploads resume PDF. FastAPI extracts text via ExamGenie's layout-aware parsing pipeline.
  2. **Multi-Agent Evaluation:** CrewAI agent (Resume Screener) compares skills against JD. A second agent (Bias Auditor) reviews evaluation notes to remove gender/age-related assumptions.
  3. **Output Generation:** Output structured JSON with score (1-100) and specific justification matching JD requirements.

### Q23. Build a coding assistant agent with GitHub integration.
**A:**
- **Design Flow:**
  - GitHub Webhook alerts FastAPI on PR creation.
  - Ingestion pulls PR git diff. Reranking fetches relevant context chunks from codebase embeddings.
  - Analysis node reviews logic, security, and formatting. Writes inline suggestions via GitHub API.
- **Safety:** Sandboxed testing (Docker) validates if suggestions compile before posting reviews.

### Q24. Design a research agent that searches, summarizes and cites sources.
**A:**
- **Architecture:**
  - User Query → Planner drafts sub-questions.
  - Tool Node calls search APIs (Tavily/Google Scholar) in parallel.
  - Synthesis node aggregates texts, extracts citation references, writes unified markdown report.
- **Grounding:** Deep research pattern similar to Nyaya-Pro case-research extension.

### Q25. Design a customer support agent that can execute refunds.
**A:**
- **Mitigation & Safety:** Must have strict transaction limits, OAuth authentication, and Human-in-the-loop authorization gates for transactions above $50. Uses Keycloak JWT verification on FastAPI backend.

### Q26. How would you implement agent memory?
**A:**
- **Memory Stack:**
  - **Short-Term:** LangGraph State checkpointer stored in Redis (conversation history buffer).
  - **Long-Term Semantic:** Embed user preferences and past interactions in Vector DB (Pinecone/FAISS).
  - **Procedural:** System prompts detailing roles and operational boundaries.

### Q27. Design an agent capable of booking flights and hotels.
**A:**
- **Design:** Multi-agent design (Flight Agent + Hotel Agent) coordinated by a Supervisor Agent in a LangGraph workflow. Prevents single agent token bloat and keeps APIs separated.

### Q28. How would you prevent infinite agent loops?
**A:**
- **Strategy:**
  1. **Strict Max Iterations Cap:** Force checkpointer to count loop iterations. Stop and alert human at N=10.
  2. **Repetition Detection:** Hash agent thoughts/actions. If identical hashes occur back-to-back, force early termination.
- **Grounding:** Built into JobPilot's CrewAI orchestrator stack.

### Q31. How would you implement tool selection?
**A:**
- **Strategy:**
  1. **Native Function Calling:** Use LLM tool schemas (`tools` parameter) which output structured JSON payload pointing to target tool.
  2. **Semantic Routing:** Retrieve relevant tools from a local embedding store if the agent has access to >50 tools, avoiding prompt window overflow.

### Q32. Design an email triage agent.
**A:**
- **Triage Flow:** Ingestion via IMAP → Intent classification (FastAPI backend) → Route to Auto-Responder (FAQ match) or Escalation queue. Grounded in OneYotta ticketing workflow.

### Q33. Design a sales outreach agent.
**A:**
- **Design:** Crawl targets via LinkedIn/Google API → CrewAI Analyzer identifies company pain points → Writer drafts personalized cold email matching company brand rules.

### Q34. How would you handle agent failures?
**A:**
- **Error Handling:** Tool output exception mapping (return traceback to LLM context to prompt self-correction). Fall back to alternative models (e.g., swap Groq with local Ollama) if API rate-limit occurs.

### Q35. Design a self-correcting agent using Reflection Pattern.
**A:**
- **Reflection Flow:** Generator drafts output → Critic evaluates draft against rules → Generator corrects → Repeat until Critic passes or max_reflection limit is reached.

### Q36. Design an AI agent for stock market research.
**A:**
- **Design:** Financial RAG (earnings calls) + Yahoo Finance API tool access for current pricing. High mathematical validation using Python code execution tools instead of raw LLM reasoning.

### Q37. How would you enforce agent guardrails?
**A:**
- **Strategy:** Input scrubbing (NeMo Guardrails), strict regex filters on outputs (PII block), and tool schema parameter validation inside FastAPI controllers.

### Q38. Design an AI agent with long-term memory.
**A:**
- **Design:** Periodic summary workers. When session ends, compress conversation history into facts, index in PostgreSQL and FAISS. Query matches long-term facts using user profile keys.

### Q39. Design an autonomous workflow execution agent.
**A:**
- **Workflow Design:** Define tasks as a DAG (Directed Acyclic Graph) in LangGraph. Executor node uses tool calling to complete tasks sequentially, updating workflow state database.

### Q40. Agent vs Workflow Automation: when would you use each?
**A:**
- **Rule of Thumb:**
  - **Workflow Automation:** If the task has clear, structured rules and sequential steps (e.g. generating an MCQ from a PDF in ExamGenie). Zero random variation allowed.
  - **Agent:** If the task requires open-ended discovery, dynamic tool usage, and self-correction (e.g. deep legal case research in Nyaya-Pro).

---
## 🤝 Section 3: Multi-Agent Systems (Questions 41–50)


### Q41. Design a multi-agent software development team.
**A:**
Here is the design for a multi-agent system automating code generation, testing, and review, leveraging my CrewAI experience in JobPilot AI.

#### 1. Clarifying Questions
- What programming languages do we support? (Python, JavaScript/TypeScript).
- How do agents communicate? (Structured JSON messaging over a unified event channel).

#### 2. High-Level Architecture (LangGraph / CrewAI)
```
+-------------------------------------------------------------------------+
|                          Product Manager Agent                          |
|        User Input -> Parses requirements -> Structured Task Spec        |
+-------------------------------------------------------------------------+
                                    |
                                    v
+-------------------------------------------------------------------------+
|                            Developer Agent                              |
|  Writes code + writes unit tests -> Writes files to sandboxed workspace  |
+-------------------------------------------------------------------------+
                                    |
                                    v
+-------------------------------------------------------------------------+
|                             Testing Agent                               |
|  Runs code in Docker container -> If error: Returns logs to Developer   |
+-------------------------------------------------------------------------+
                                    |
                                    v
+-------------------------------------------------------------------------+
|                             Reviewer Agent                              |
|           Compares spec vs output -> Approves / Returns modifications   |
+-------------------------------------------------------------------------+
```

#### 3. State Management & Coordination
- LangGraph orchestrates the DAG. The state dictionary maintains code files, test logs, requirements, and reflection retry count.
- Communications are kept strictly structured using Pydantic types to prevent formatting decay.

#### 4. Security & Isolation
- **Execution Sandbox:** The Testing Agent calls a subprocess runner directing execution to isolated short-lived Docker containers. Networking is disabled, and storage is ephemeral. This protects our system from malicious code execution.

#### 5. Tradeoffs
- **Sequential CrewAI vs LangGraph State Machine:** CrewAI is easy to set up but unpredictable for loops. I chose LangGraph conditional edges because it allows structured retry logic (Developer -> Tester -> Developer loop) up to a max retry limit.

### Q42. Design a planner-executor-reviewer architecture.
**A:**
🏷️ *This architecture is grounded in my multi-agent ATS optimizer pipeline in JobPilot AI, coordinating specialized content analysis, generation, and compliance checking nodes.*

#### 1. Clarifying Questions
- Can the reviewer change plans or just reject outputs? (Both: Reviewer suggests plan modifications back to the Planner).
- What model matches best? (Planner/Reviewer use GPT-4o; Executor uses fast Gemini Flash / Groq APIs).

#### 2. Component Design & Coordination
```
User Goal -> [Planner Node] (Generates checklist DAG)
                   | ^
                   v | (Refinement Loop)
             [Executor Node] (Executes task 1..N using tools)
                   |
                   v
             [Reviewer Node] (Validates checklist -> Done / Refinement)
```
- **Planner:** Formulates steps. For JobPilot, it identifies missing resume achievements matching target jobs.
- **Executor:** Focuses on writing content. Uses Groq APIs for sub-second text generation.
- **Reviewer:** Scores compliance. Checks if ATS keyword requirements are met without introducing hallucinated experience. Returns recommendations to Executor if checks fail.

#### 3. Tradeoffs
- Decoupled roles prevent LLM instruction drift, raising quality by ~20%. However, sequential generation steps increase latency. I resolved this in UI using streaming and partial step disclosure.

### Q43. Design a multi-agent customer support platform.
**A:**
- **Orchestration:** LangGraph supervisor distributes tickets to specialized sub-agents: Billing Agent, Tech Support Agent, and Escapes Coordinator. State tracking matches active ticket IDs, Keycloak permissions, and user token context.

### Q44. When would you use multiple agents instead of one?
**A:**
- **Decision Criteria:** Use multi-agent systems when: (1) The task requires distinct, conflict-prone instructions (e.g. Writer vs. Editor), (2) The prompt size exceeds LLM attention capacity, or (3) Different components require specialized tool access (e.g., local python running vs web searching).

### Q45. Design an AI investment advisory team.
**A:**
- **Design:** CrewAI team: (1) News Analyst (searches Google News/Sec), (2) Quantitative Calculator (runs math in python sandbox), (3) Portfolio Planner (rebalances based on user risk score). Outputs compiled as PDF reports via FastAPI.

### Q46. How would agents communicate with each other?
**A:**
- **Communication Flow:** Standardized JSON contracts over Redis Pub/Sub channels (asynchronous event architecture) or direct state mutations in a shared LangGraph memory checkpointer.

### Q47. Design conflict resolution among agents.
**A:**
- **Design:** A dedicated Mediator Node evaluates contradictory suggestions. If the Quantitative agent says 'buy' and the Risk agent says 'sell', the Mediator parses reasoning trees against compliance rules to make the final decision.

### Q48. Design agent orchestration using LangGraph.
**A:**
- **Implementation:** Define state schema → Add nodes (python helper functions mapping to LLM calls) → Define edges (transitions) and conditional edges (e.g. router mapping output to next node based on JSON state flag) → Compile graph.

### Q49. Design a supervisor agent architecture.
**A:**
- **Design:** A central Supervisor LLM parses user query, decides which sub-agent is active next, and routes to it. Sub-agent completes task and returns output to supervisor. Cycle terminates when Supervisor returns `__end__` token.

### Q50. How would you evaluate multi-agent performance?
**A:**
- **Metrics:** containment rate, average steps-to-resolution, cost per successful run, task completion rate, and semantic consistency of state transitions (flagging looping cycles).

---
## ⚙️ Section 4: LLM Infrastructure & Scalability (Questions 51–60)


### Q51. Design ChatGPT for 100M users.
**A:**
Here is the system architecture to support 100 million active users (millions of concurrent sessions), combining web optimization (NgRx, Web Workers from my Yotta projects) with advanced LLMOps pipelines.

#### 1. Clarifying Questions
- What is the average traffic? (Assume 10,000 requests per second peak).
- What are the token dimensions? (Average input 100 tokens, output 300 tokens).

#### 2. High-Level Architecture
```
Client (SSE Stream) -> CDN -> API Gateway / Load Balancer (Kong)
                                    |
                                    v
                            FastAPI Routing
                                    |
                    +---------------+---------------+
                    |                               |
                    v                               v
              vLLM Cluster                    Redis Cache
         (Continuous Batching,              (Semantic Cache)
          PagedAttention, GQA)
```

#### 3. Serving & Optimization Stack
- **vLLM Cluster:** Deploy models (e.g. Llama 3 70B sharded via Tensor Parallelism across 8x H100 GPUs) on Kubernetes.
- **PagedAttention:** Manages KV cache memory fragmentation, allowing 2-4× higher throughput.
- **Continuous Batching:** Dynamic request batching at iteration level instead of sequence level, maximizing GPU utilization.

#### 4. Scalability & Latency Optimization
- **Edge Streaming:** Use Server-Sent Events (SSE) to stream tokens instantly as they are generated. Minimizes perceived latency.
- **Prompt Prefix Caching:** Cache the KV cache of system prompts. Saves GPU compute cycles.
- **Global Multi-Region Routing:** API Gateway routes traffic to closest active data center hosting GPU clusters, optimizing connection handshakes.

#### 5. Tradeoffs
- **Self-Hosted vLLM vs API:** Self-hosting H100 GPU clusters is highly expensive upfront but saves millions of dollars over commercial APIs at 100M user scale.

### Q52. How would you reduce inference cost by 50%?
**A:**
🏷Header: *This is the cost control strategy I implemented for Nyaya-Pro and JobPilot AI to limit token costs in production.*

#### 1. Clarifying Questions
- What are the primary cost drivers? (Large system prompt context, model size, output verbosity).

#### 2. Cost Reduction Methods Matrix
```
+----------------------+-------------------+-----------------------+
| Method               | Cost Reduction %  | Quality Impact        |
+----------------------+-------------------+-----------------------+
| Prompt Caching       | 30 - 45%          | None                  |
| Semantic Cache (Redis)| 20 - 40%         | None (exact match)    |
| Quantization (AWQ)   | 50%               | Negligible (< 1%)     |
| Router (Model Cascade)| 40 - 50%          | Selective             |
+----------------------+-------------------+-----------------------+
```

#### 3. Detailed Actions
- **Model Cascading:** Run an classifier node. Simple queries are routed to cheap models (Gemini Flash / GPT-4o-mini); only complex logic goes to GPT-4o / Claude Sonnet.
- **Quantization:** Compile open-source models using AWQ (Activation-aware Weight Quantization) or GPTQ into 4-bit formats. This allows running a large model (e.g. 70B) on a single GPU node instead of requiring multiple nodes.
- **Aggressive Cache Implementation:** Run semantic caching in Redis. System prompts use API-level prompt caching.

#### 4. Tradeoffs
- Quantization can cause slight degradation in highly structured coding / math tasks. I deploy unquantized models for math nodes and quantized models for conversational chat nodes.

### Q53. Design a token-efficient architecture.
**A:**
- **Optimization Strategy:** System prompts minimized using compact templates. Trim user history dynamically using sliding window summarization. metadata returned as compressed IDs. Grounded in JobPilot CrewAI token pruning.

### Q54. Design LLM serving on Kubernetes.
**A:**
- **Infrastructure:** K8s cluster utilizing KEDA (Kubernetes Event-driven Autoscaling) to scale pod replicas based on active queue metrics. Triton/vLLM containers mapped to GPU resource limits (`nvidia.com/gpu`).

### Q55. Design multi-region LLM deployment.
**A:**
- **Architecture:** Active-active multi-region deployment. Route requests using Cloudflare GeoDNS. DB states (user sessions) replicated using global CockroachDB / Redis Enterprise clusters.

### Q56. How would you implement KV-cache optimization?
**A:**
- **Strategy:** GQA (Grouped Query Attention) configuration to share KV heads, PagedAttention in vLLM, and RadixAttention to share KV cache across multiple concurrent requests with identical prefixes.

### Q57. Design load balancing for LLM requests.
**A:**
- **Balancing Design:** Balance requests based on **GPU Memory Utilization** and **Queue Depth** instead of simple round-robin (which fails due to highly variable output token lengths).

### Q58. Design a fallback model strategy.
**A:**
- **Strategy:** If Groq API rate-limits, catch error and fall back to local Ollama Llama container instance in FastAPI backend. Circuit breaker pattern shuts off failed routes for 60s.

### Q59. How would you optimize GPU utilization?
**A:**
- **Strategy:** Continuous batching, speculative decoding (small draft model outputs matched by target model), and scheduling model ingestion using FlashAttention kernels.

### Q60. Design an LLM platform supporting multiple models.
**A:**
- **Design:** Unified API gateway proxying requests. Internal routers map requests to target model endpoints (Ollama, Groq, Gemini) based on standard request JSON format.

---
## 📊 Section 5: Evaluation & Monitoring (Questions 61–70)


### Q61. Design an evaluation framework for RAG.
**A:**
Here is the evaluation pipeline to continuously assess and test RAG quality, grounded in my Nyaya-Pro development stack.

#### 1. Clarifying Questions
- Is evaluation run online (production traffic) or offline (CI/CD)? (Both: Offline regression testing + online telemetry sampling).
- What models rate quality? (We use GPT-4o as judge, plus automated semantic distance metrics).

#### 2. RAGAS Metrics Mapping
```
               +-------------------+-------------------+
               | Context Precision | Context Recall    | (RETRIEVAL)
               +-------------------+-------------------+
                         |                   |
                         v                   v
               +-------------------+-------------------+
               |   Faithfulness    | Answer Relevance  | (GENERATION)
               +-------------------+-------------------+
```

#### 3. Continuous Evaluation Data Pipeline
1. **Golden Dataset Creation:** Construct a version-controlled suite of 200 query-context-response triplets (curated by domain experts).
2. **CI/CD Integration:** Trigger tests on prompt or embedding updates. Build a python runner using `Ragas` to calculate scores.
3. **Online Telemetry:** Log production inputs, retrieved context chunks, and output responses to Arize Phoenix database via open-telemetry tracing.
4. **Dashboard Analysis:** Display regression trends. If Faithfulness drops < 90%, flag build.

#### 4. Tradeoffs
- **LLM-as-a-judge vs Human Evaluation:** LLM-as-a-judge is cheap, instant, and scalable, but prone to self-model bias. We combine it with human sampling (auditing 1% of flagged low-score cases).

### Q68. Design tracing for agent execution.
**A:**
🏷️ *This traces complex multi-turn transitions. I designed similar logging flows to debug JobPilot's CrewAI agent nodes.*

#### 1. Clarifying Questions
- What tracing framework connects? (OpenTelemetry standard, mapping to LangSmith/Phoenix).

#### 2. Tracing DAG Flow
```
Parent Run ID (Agent Execution Session)
├── Step 1 (Thought Process Node) -> Latency, tokens
├── Step 2 (Tool Call: Web Search) -> Input params, exit code, output payload
└── Step 3 (Self-Reflection Node) -> Evaluation score, corrective instructions
```
- **Trace Identifiers:** Every agent cycle generates a unique `Session-ID` and passes parent-span IDs down tool executions.
- **Metrics Logs:** Capture Time-to-First-Token, tool execution latency, and token consumption count per node.
- **Failure Isolation:** Log exact stack trace in the span context when tool failures occur. This alerts developers directly via webhook notifications.

#### 3. Tradeoffs
- Logging full inputs/outputs of LLM runs can introduce storage overhead and compliance issues (PII leak). We hash payload strings before telemetry export in production pipelines.

### Q62. How would you measure hallucination rate?
**A:**
- **Methodology:** Use Natural Language Inference (NLI) model or LLM-as-a-judge to compare target sentences in output response against source documents. Score calculated as percentage of unsupported statements.

### Q63. Design online vs offline evaluation.
**A:**
- **Design:**
  - **Offline:** Run test runner (RAGAS/Golden dataset) on pull requests. Strict gating blocks deployment if benchmarks degrade.
  - **Online:** Telemetry tracking, implicit user signals (copy clicks, thumbs up), and auditing feedback forms in production.

### Q64. How would you evaluate agent quality?
**A:**
- **Metrics:** Target-state path completeness, tool call correctness (precision/recall of calls), task success rate, and loops count.

### Q65. Design A/B testing for prompts.
**A:**
- **A/B Design:** Route 10% traffic to route B using feature flags. Log outputs along with prompt-version hash tag. Compare downstream metrics: task completion and error rate.

### Q66. Design observability for LLM applications.
**A:**
- **Stack:** OpenTelemetry tracing on FastAPI app, exporting logs to Prometheus (latency, cost) and LangSmith/Phoenix (trace visualization).

### Q67. How would you monitor model drift?
**A:**
- **Monitoring:** Track output length distribution, vocabulary diversity, response confidence scores, and user rating trends. Alert if average metrics drift > 10% over weekly intervals.

### Q69. How would you identify retrieval failures?
**A:**
- **Strategy:** Flag cases where vector cosine similarity is < 0.65, or where LLM judge identifies that the retrieved context is irrelevant to the query.

### Q70. Design a human-in-the-loop review system.
**A:**
- **Design:** Flag low-confidence outputs, route them to an Angular-based admin queue (Label Studio API integration), where developers/experts audit and correct answers.

---
## 🛡️ Section 6: AI Safety & Governance (Questions 71–80)


### Q71. Design a safe enterprise chatbot.
**A:**
Here is the safety architecture to prevent jailbreaks, prompt injection, and data leakage, grounded in my security-focused enterprise deployments.

#### 1. Clarifying Questions
- What security certifications do we require? (SOC2, GDPR, HIPAA compliance).
- How do we handle PII? (Mask PII before processing queries).

#### 2. Multi-Layer Safety Architecture
```
User Input -> (1) Input Guard (Llama Guard / regex filter for injections)
           -> (2) Authentication & RBAC Check (Keycloak JWT)
           -> (3) Vector Search (Scope-restricted metadata query)
           -> (4) LLM Processing (System-level constraints)
           -> (5) Output Guard (Scans output for PII leakage / safety)
           -> User Output
```

#### 3. Core Safety Modules
- **Llama Guard Engine:** Host a local `Llama-Guard-3-8B` model as a filter before routing query. Screens for violence, hate speech, and injection attacks.
- **RBAC Filtering:** Hard-coded security filters applied to vector searches, blocking unauthorized document reads (see RAG designs).
- **Output PII Scrubber:** Local regex/Presidio processor validates outputs, masking matching credit card, SSN, or email configurations.

#### 4. Tradeoffs
- Adding safety classifiers increases latency (~100-150ms per guard), but is mandatory to protect enterprise databases from public disclosure.

### Q74. Design role-based access control in RAG.
**A:**
🏷️ *This is the security architecture I implemented for Nyaya-Pro to ensure that specific acts and corporate records were filtered by role permissions.*

#### 1. Clarifying Questions
- Can roles change in real-time? (Yes: auth tags mapped inside the token validation flow).

#### 2. Secure RAG Architecture
```
Client (with JWT) -> FastAPI (Decodes JWT allowed-roles: ["finance"])
                             |
                             v
                 Query Vector + Metadata Filter
              `"allowed_roles": {"$in": ["finance"]}`
                             |
                             v
                 Vector Database (Pinecone/Milvus)
              (Only returns matching security vectors)
```
- **Ingestion Labeling:** During indexing, every chunk contains metadata tag: `{"allowed_roles": ["role_name_1", "role_name_2"]}`.
- **Serving Enforcement:** FastAPI decodes JWT token roles and injects MongoDB-style query filters directly into Vector DB call. The DB executes filtering at index-traversal level. This guarantees that unauthorized chunks never enter LLM memory context.

#### 3. Tradeoffs
- **Post-filtering vs In-index filtering:** Post-filtering (retrieving top-50 vectors, then checking security in code) is highly insecure and drops recall. We always enforce in-index pre-filtering to guarantee security and maintain retrieval recall quality.

### Q72. How would you prevent prompt injection attacks?
**A:**
- **Defense Strategy:** Use XML tagging inside system prompts (e.g. `<user_input>{input}</user_input>`), instruct the model to ignore instructions outside the tags, and use input classifier models (Llama Guard).

### Q73. Design guardrails for an AI banking assistant.
**A:**
- **Guardrails:** Pre-filter transaction terms, enforce PII masking, isolate financial calculations to standard backend APIs (not LLM generation), and validate user authorization tokens (Keycloak) for every request.

### Q75. How would you secure tool-calling agents?
**A:**
- **Strategy:** Parameter schema sanitization (Pydantic validation), strict REST endpoint authorization checks inside tool code, and executing file/shell tools in isolated Docker environments.

### Q76. Design compliance architecture for healthcare AI.
**A:**
- **Design:** Local hosting of LLMs on private enterprise clouds (Yotta GPU nodes) to prevent HIPAA compliance leaks, de-identifying data using specialized NLP tools, and tracking database audit logs.

### Q77. How would you detect jailbreak attempts?
**A:**
- **Strategy:** Keep an adversarial injection dataset (dan, roleplay attacks) for regression tests. Deploy a local classification helper trained specifically to flag common jailbreak sequences.

### Q78. Design data governance for GenAI.
**A:**
- **Governance:** Establish clear rules mapping which databases can feed vector indexes. Scrub data sources for user PII and confidential terms, and automate vector index deletion on customer exit.

### Q79. Design AI audit logging.
**A:**
- **Logging Design:** Stream database transactions (Kafka) logging: user ID, prompt metadata hash, model, target document sources, response hash, and validation metrics to a write-once secure database.

### Q80. How would you implement responsible AI controls?
**A:**
- **Controls:** Bias testing on dataset generation, explicit disclaimer alerts inside frontend, human review gates, and drift monitoring.

-- ---
## 🔧 Section 7: Fine-Tuning & Model Optimization (Questions 81–90)


### Q84. Design a LoRA-based fine-tuning pipeline.
**A:**
Here is the pipeline design to execute Low-Rank Adaptation (LoRA) fine-tuning of open-source models, grounding in my ExamGenie AI synthetic data extraction loops.

#### 1. Clarifying Questions
- What base model do we use? (Llama-3-8B-Instruct).
- What is the target hardware? (NVIDIA A100/H100 80GB GPU nodes).

#### 2. Fine-Tuning Pipeline Flow
```
Raw Data -> Ingestion (Data Cleaning) -> Synthetic Data Gen (via GPT-4)
                                     -> Format to Instruction JSON
                                     -> PEFT (LoRA rank=16, alpha=32)
                                     -> Validation (Weights & Biases)
                                     -> Merge Adapter -> vLLM Serving
```

#### 3. Ingestion & Hyperparameter Tuning
- **Data Prep:** Format data into clean instruction-response pairs matching the chat template format.
- **PEFT Configuration:**
  - LoRA target modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`.
  - Rank $r=16$, scale factor $\alpha=32$ (reduces parameters trained to < 1%).
  - Quantization: QLoRA (4-bit quantization of base model) if memory optimization is required.

#### 4. Evaluation & Merge
- Monitor loss curves on Weights & Biases. Watch for training loss drop without validation loss divergence (prevent overfitting).
- Merge target LoRA adapters back to base model weights, export model weights as GGUF / SafeTensors for vLLM container serving.

#### 5. Tradeoffs
- **LoRA vs Full Fine-Tuning:** LoRA saves 90% in hardware costs and avoids catastrophic forgetting, but might limit final style adaptation compared to full parameters updates. I recommend LoRA for formatting and styling.

### Q81. When would you choose Fine-Tuning over RAG?
**A:**
- **Decision Framework:** Choose Fine-Tuning when you need to adapt style, formatting, tone, or code syntax (e.g. formatting ExamGenie MCQ output). Choose RAG when you need to inject factual knowledge that changes frequently (e.g. Nyaya-Pro legal updates).

### Q82. Design a domain-specific medical chatbot.
**A:**
- **Design:** Quantized Llama 3 medically instruction-tuned on SNOMED datasets using QLoRA, coupled with real-time medical guideline RAG (PubMed) to prevent hallucinations.

### Q83. How would you fine-tune with limited data?
**A:**
- **Strategy:** Generate synthetic datasets using GPT-4, execute Low-Rank Adaptation (LoRA) targeting only linear projection modules to avoid overfitting, and apply strict early stopping.

### Q85. How would you avoid catastrophic forgetting?
**A:**
- **Strategy:** Mix 10% general-instruction dataset (e.g. ShareGPT) into domain-specific datasets during training. This keeps the model's basic conversational skills active.

### Q86. Design continual learning architecture.
**A:**
- **Design:** Maintain model adapters (LoRA) updated weekly. The base model weights remain frozen, while specific task-based adapters are re-trained and hot-swapped dynamically at inference.

### Q87. Fine-tuning vs Prompt Engineering tradeoffs?
**A:**
- **Tradeoffs:** Prompt engineering is fast and zero cost to test but consumes context tokens. Fine-tuning reduces token consumption but requires GPU training costs and setup complexity.

### Q88. Design a custom LLM for legal documents.
**A:**
- **Design:** Continue pre-training (domain adaptation) on legal corpora using a small learning rate, followed by LoRA instruction tuning matching citation requirements. Connect to FAISS legal index.

### Q89. How would you evaluate a fine-tuned model?
**A:**
- **Strategy:** Evaluate perplexity on a held-out validation set. Run downstream tasks on benchmark datasets (MMLU, HumanEval) and compare outputs using LLM-as-a-judge.

### Q90. Design model versioning and rollback.
**A:**
- **Strategy:** Save model weights as immutable SafeTensors files indexed in MLflow. API gateway routing handles blue-green deployment, switching traffic to old model version tags if error rates spike.

---
## 🏢 Section 8: Real FAANG-Level Design Questions (Questions 91–100)


### Q91. Design ChatGPT from scratch.
**A:**
Here is the end-to-end design for training and serving an interactive conversational LLM at global scale.

#### 1. Pre-training Phase
- **Dataset:** Trillion-token text/code corpus (CommonCrawl, Github, Books). Deduplicated and scrubbed.
- **Architecture:** Decoder-only transformer. SwiGLU activations, RMSNorm, Rotary Embeddings (RoPE), and Grouped-Query Attention (GQA).
- **Infrastructure:** Megatron-LM distributed training cluster using data, tensor, and pipeline parallelism across 4000+ A100/H100 GPUs.

#### 2. Instruction Tuning (SFT) & Alignment (RLHF)
- **Supervised Fine-Tuning:** Instruction tune on 100K+ high-quality human demonstrations.
- **Alignment (RLHF/GRPO):** Train a reward model on user preference pairs, then run policy optimization (GRPO/PPO) to align responses with helpfulness and safety guidelines.

#### 3. Production Serving Infrastructure
```
User Client -> CDN -> Kong Gateway (Rate Limiter)
                            |
                            v
                    FastAPI Orchestration
                            |
               +------------+------------+
               |                         |
               v                         v
         vLLM Engine (GPUS)        Redis (Sessions)
       (Continuous Batching,
        PagedAttention)
```

#### 4. Tradeoffs
- **Pre-training from scratch vs. API wrapper:** Pre-training gives 100% control but costs millions of dollars. For enterprise applications, fine-tuning pre-trained base models (Llama) is the standard recommendation.

### Q92. Design GitHub Copilot.
**A:**
Here is the design for an inline code completion assistant integrated into the IDE, focusing on sub-200ms latency constraints.

#### 1. Clarifying Questions
- What is the latency target? (Completion options must display within 150-200ms of user typing pause).
- How do we capture context? (Analyze open editor tabs, file names, import structures).

#### 2. Core Code Context Gathering Pipeline
```
IDE Plugin (Tracks Cursor Typing & Open Tabs)
                   |
                   v
     Fast Context Builder (Local)
 (Pulls surrounding lines + active symbols)
                   |
                   v
     API Gateway -> Triton Inference Server
      (Speculative Decoding with CodeLlama-13B / StarCoder)
                   |
                   v
IDE Client (Renders shadow text completion)
```

#### 3. Latency Mitigation Design
- **Speculative Decoding:** Deploy a tiny draft model (e.g. StarCoder-1B) in parallel with the main verifier model (StarCoder-15B) on Triton. The draft model generates candidates fast; the target model verifies them in parallel. This cuts latency by 2-3×.
- **Context Window Trimming:** Only send the active file context plus matching signature files retrieved via quick regex search (avoiding heavy vector queries during typing loops).

#### 4. Tradeoffs
- **Large vs Small Model:** A large model (70B) generates highly complete scripts but fails our 200ms latency budget. We use 1B-15B scale models optimized specifically for code completions.

### Q93. Design Perplexity AI.
**A:**
🏷️ *This is the direct architectural extension of Nyaya-Pro's hybrid retrieval and citation rendering stack.*

#### 1. High-Level Search & Answer Flow
```
User Query -> Query Expansion & Search Planner (Gemini Flash)
                       |
                       v
     Trigger Parallel Web Searches (Tavily/Google Search API)
                       |
                       v
      Scrape Search Result Snippets -> Temporary In-Memory FAISS index
                       |
                       v
     Hybrid Search + Reranker (Top 5 target text chunks)
                       |
                       v
     LLM Generation with Citation mapping -> SSE Stream to User
```

#### 2. Key Components
- **Query Planner:** Translates user questions into search keywords (e.g., 'Perplexity architecture' → 'Perplexity AI system design engine').
- **Scraper Engine:** Async workers fetch target HTML pages, parse raw text, and extract clean semantic chunks.
- **Verification Engine:** Validates that generated answers are grounded in retrieved snippets (RAGAS framework).

#### 3. Tradeoffs
- Scraping takes time. We stream partial results (displaying 'Scraping sources...', then 'Reranking...') to improve perceived user experience.

### Q95. Design Google NotebookLM.
**A:**
🏷️ *This builds on the layout-aware PDF MCQ generation pipeline I designed for ExamGenie AI, scaling it to multi-source context indexing.*

#### 1. Ingestion & Layout Parsing
- Upload documents (PDFs, docs, notes) → FastAPI backend.
- Parse documents preserving structure (headings, tables, page breaks) to capture contextual chunks.
- Generate document summary outlines immediately to help the user navigate.

#### 2. Context Indexing & Multi-Doc RAG
```
Uploaded Files -> Chunk -> Generate Embeddings -> Local User Index
                                                  (FAISS / Pinecone)
```
- **Radix Attention & Prompt Caching:** Load active files into LLM context. Since the user queries are session-specific, we use prompt prefix caching to keep document representations warm, saving ~70% cost.
- **Audio Generation (Podcast creation):** Convert document outlines to dialog script using an LLM, then feed to a Multimodal Audio generator (Kokoro-TTS / AudioCraft) to produce interactive audio files.

#### 3. Tradeoffs
- Storing separate indexes per user notebook requires efficient multi-tenant vector database isolation. We use Pinecone multi-tenant namespaces to keep files isolated and secure.

### Q94. Design an AI-powered search engine.
**A:**
- **Design:** Hybrid search engine. Core inverted index (Elasticsearch) + dense vector index (Milvus). Rerank results via Cross-Encoder. Summary generator writes direct answers with citations. Similar to Perplexity design.

### Q96. Design an AI coding interviewer.
**A:**
- **Design:** State machine (LangGraph): (1) Prompt Node asks question, (2) Code Evaluator Node parses input via compiler APIs, (3) Hint Router Node decides if applicant is stuck and suggests hints without giving solution. Renders response in Angular terminal UI.

### Q97. Design a real-time meeting summarization system.
**A:**
- **Design:** Ingestion fetches audio streams → Web Worker pushes audio chunks to Whisper API for transcription → Summarizer agent maps text to agenda items → Output stored in PostgreSQL. Re-rendered in UI using NgRx state management.

### Q98. Design a YouTube video understanding assistant.
**A:**
- **Design:** Fetch video transcript. If transcript is missing, extract audio via worker, run STT (Whisper). Sample frames every 5s, pass transcript + frames to multimodal LLM (Gemini 2.5 Flash) with prefix prompt caching.

### Q99. Design an AI-powered tax advisor.
**A:**
- **Design:** RAG system querying tax codes, connected to a sandboxed Python execution tool to calculate financial sums. Pre-filtered by user permissions and tax files. Same as tax advisory agent Q29.

### Q100. Design an autonomous software engineer agent.
**A:**
- **Design:** Fully agentic software developer (equivalent to Q41). LangGraph coordinates requirements parsing, file structure creation, code writing, and dockerized test execution loops, outputting GitHub PR diffs.

---
## 🏆 Most Frequently Asked in 2026 Senior GenAI Interviews

### 1. Design Enterprise RAG
- **Summary:** Reference **Q1** and **Q3**. Leverage hybrid search (BM25 + Dense vector), cross-encoder rerankers, metadata filtering for RBAC (Keycloak mapping), and continuous telemetry using RAGAS.

### 2. Design Agentic AI System
- **Summary:** Reference **Q21**. Build structured state machines using LangGraph. Enforce Human-in-the-loop validation for financial/database writes, validation check nodes, and loop countermeasures (iteration capping).

### 3. Design Multi-Agent Architecture
- **Summary:** Reference **Q41** and **Q42**. Coordinated supervisor models routing tasks to specialized agents (PM, Dev, Tester) passing Pydantic structured payloads. Docker isolation for code runs.

### 4. Design AI Copilot
- **Summary:** Reference **Q92**. Real-time code/text completion engines. Sub-200ms target latency achieved using speculative decoding, light local context builders, and custom quantized models served on Triton.

### 5. Design AI Search Engine
- **Summary:** Reference **Q93** and **Q94**. Direct answer engines pulling live data via parallel web searches, building temporary vector databases, reranking, and generating citation-attributed answers.

### 6. Design Evaluation Framework
- **Summary:** Reference **Q61**. Decoupled offline evaluations (Golden datasets run on PR triggers in CI/CD) and online tracking (OpenTelemetry collecting spans to Arize Phoenix monitoring databases).

### 7. Design AI Safety Layer
- **Summary:** Reference **Q71** and **Q74**. Multi-guard architecture comprising input guards (Llama Guard), strict DB metadata role validation (RBAC check), system prompt boundaries, and output PII scrubbers.

### 8. Design AI Workflow Automation
- **Summary:** Reference **Q40**. Map structured, predictable flows into deterministic state machine graphs. High predictability, low hallucination risk compared to open-ended agentic loops.

### 9. Design LLM Platform at Scale
- **Summary:** Reference **Q51** and **Q52**. Large-scale inference serving hosting open-source models using vLLM (continuous batching, GQA, PagedAttention) dynamically scaled on Kubernetes clusters.

### 10. Design Autonomous Research Agent
- **Summary:** Reference **Q24**. Planner agents spawning parallel search workers, scraping target domains, structuring markdown reports, and verifying citations using NLI classifiers.